# 07 Export Results (Local Files Only)

This notebook consolidates outputs from previous notebooks into final local files.

Mode:
1. **Local mode:** saves one final analytics package CSV/JSON.

No database upload is used in this project workflow.

In [ ]:
from pathlib import Path
import json
import uuid
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

In [ ]:
BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
FINAL_DIR = OUTPUT_DIR / "final_package"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "final_clusters": OUTPUT_DIR / "final_customer_clusters_with_anomalies.csv",
    "final_clusters_fallback": OUTPUT_DIR / "final_customer_clusters.csv",
    "metrics": OUTPUT_DIR / "clustering_metrics_comparison.csv",
    "selection_report": OUTPUT_DIR / "final_model_selection_report.csv",
    "selected_model_json": OUTPUT_DIR / "final_selected_model.json",
    "assoc_rules_top": OUTPUT_DIR / "association_rules_top.csv",
    "assoc_rules_all": OUTPUT_DIR / "association_rules_all.csv",
    "anomaly_summary": OUTPUT_DIR / "anomaly_summary.csv",
}

for k, p in paths.items():
    print(k, "=>", p.exists(), p)

final_clusters => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\final_customer_clusters_with_anomalies.csv
final_clusters_fallback => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\final_customer_clusters.csv
metrics => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\clustering_metrics_comparison.csv
selection_report => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\final_model_selection_report.csv
selected_model_json => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\final_selected_model.json
assoc_rules_top => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\association_rules_top.csv
assoc_rules_all => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\association_rules_all.csv
anomaly_summary => True S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\anomaly_summary.csv


In [ ]:
# Load required artifacts
base_path = paths["final_clusters"] if paths["final_clusters"].exists() else paths["final_clusters_fallback"]
if not base_path.exists():
    raise FileNotFoundError("Run notebooks 04 and 05 first to generate final cluster files.")

final_clusters = pd.read_csv(base_path)
metrics = pd.read_csv(paths["metrics"]) if paths["metrics"].exists() else pd.DataFrame()
selection_report = pd.read_csv(paths["selection_report"]) if paths["selection_report"].exists() else pd.DataFrame()
assoc_top = pd.read_csv(paths["assoc_rules_top"]) if paths["assoc_rules_top"].exists() else pd.DataFrame()
anom_summary = pd.read_csv(paths["anomaly_summary"]) if paths["anomaly_summary"].exists() else pd.DataFrame()

if paths["selected_model_json"].exists():
    selected_meta = json.loads(paths["selected_model_json"].read_text(encoding="utf-8"))
else:
    selected_meta = {}

print("final_clusters:", final_clusters.shape)
print("metrics:", metrics.shape)
print("selection_report:", selection_report.shape)
print("assoc_top:", assoc_top.shape)
print("anom_summary:", anom_summary.shape)

final_clusters: (15, 31)
metrics: (4, 7)
selection_report: (4, 15)
assoc_top: (25, 14)
anom_summary: (1, 6)


In [ ]:
# Build final package summary
run_id = str(uuid.uuid4())
created_at = datetime.now().astimezone().isoformat()

selected_model = selected_meta.get("selected_model")
if not selected_model and "selected_model" in final_clusters.columns:
    selected_model = str(final_clusters["selected_model"].dropna().iloc[0]) if final_clusters["selected_model"].notna().any() else "Unknown"
selected_model = selected_model or "Unknown"

summary_payload = {
    "run_id": run_id,
    "created_at_utc": created_at,
    "selected_model": selected_model,
    "total_customers": int(len(final_clusters)),
    "total_clusters": int(final_clusters["cluster_label"].nunique()) if "cluster_label" in final_clusters.columns else None,
    "anomaly_count": int(final_clusters["anomaly_flag"].sum()) if "anomaly_flag" in final_clusters.columns else None,
    "anomaly_rate": float(final_clusters["anomaly_flag"].mean()) if "anomaly_flag" in final_clusters.columns else None,
    "top_rules_count": int(len(assoc_top)),
}

summary_payload

C:\Users\sathe\AppData\Local\Temp\ipykernel_22692\694928379.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  created_at = datetime.utcnow().isoformat() + "Z"


{'run_id': 'add94e96-c13b-46c3-970d-da86785a1f34',
 'created_at_utc': '2026-05-08T16:37:52.124662Z',
 'selected_model': 'DBSCAN',
 'total_customers': 15,
 'total_clusters': 1,
 'anomaly_count': 1,
 'anomaly_rate': 0.06666666666666667,
 'top_rules_count': 25}

In [ ]:
# Save local final package files
final_clusters_path = FINAL_DIR / "final_customer_clusters.csv"
metrics_path = FINAL_DIR / "clustering_metrics.csv"
selection_path = FINAL_DIR / "model_selection_report.csv"
assoc_top_path = FINAL_DIR / "association_rules_top.csv"
summary_json_path = FINAL_DIR / "run_summary.json"

final_clusters.to_csv(final_clusters_path, index=False)
if not metrics.empty:
    metrics.to_csv(metrics_path, index=False)
if not selection_report.empty:
    selection_report.to_csv(selection_path, index=False)
if not assoc_top.empty:
    assoc_top.to_csv(assoc_top_path, index=False)

summary_json_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print("Saved local package files in:", FINAL_DIR)
for p in sorted(FINAL_DIR.glob("*")):
    print("-", p.name)

Saved local package files in: S:\cafeteria\cafeteria-iq\backend\notebooks\outputs\final_package
- association_rules_top.csv
- clustering_metrics.csv
- final_customer_clusters.csv
- model_selection_report.csv
- run_summary.json


## Local-Only Workflow

Database export is intentionally disabled.

All results needed for your project are already saved under:
- `outputs/final_package/`
- `outputs/*.csv`

Use these files directly in reports, dashboards, or presentations.

In [ ]:
# Local-only notebook.
# No external database client installation is required.

In [ ]:
# Local-only setup confirmation
print("Database step skipped. Local CSV export pipeline is active.")

SUPABASE_URL set: False
SUPABASE_SERVICE_ROLE_KEY set: False


In [ ]:
# Local helper: show generated files quickly
generated_files = sorted([p.name for p in FINAL_DIR.glob("*")])
print("Generated files:")
for name in generated_files:
    print("-", name)

In [ ]:
# Final local status message
print("Local export complete. No database operations were performed.")
print("Use outputs/final_package files for your final submission.")

## Next Notebook

Move to `08_final_business_insights.ipynb` to generate final presentation insights, persona narratives, and recommendation slides content.